# Vaani KWS - V2 Dataset Preparation

This notebook discovers all WAV files from dataset/Dataset_v1 at runtime, calculates 70/15/15 split targets from the discovered counts, and prepares model_v2/data without modifying the source dataset.

- Positives: per-speaker, SHA-aware dynamic split
- Silence and background: source-aware dynamic split
- Speech Commands: every discovered file, split per category
- All copied paths retain their source-relative directories


## 1. Configuration

In [ ]:
# ============================================================
# SECTION 1 - CONFIGURATION
# ============================================================

import os
import sys
import json
import wave
import shutil
import hashlib
import re
import math
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)

EXPECTED_SR = 16000
EXPECTED_CHANNELS = 1
EXPECTED_SAMPLE_WIDTH = 2
EXPECTED_DURATION_S = 1.0
DURATION_TOLERANCE_S = 0.05

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

LABEL_POSITIVE = 1
LABEL_NEGATIVE = 0

SPLITS = ["train", "validation", "test"]
CATEGORIES = [
    "positive",
    "negative_silence",
    "negative_background",
    "negative_speech_commands",
]
SPLIT_RATIOS = {
    "train": TRAIN_RATIO,
    "validation": VAL_RATIO,
    "test": TEST_RATIO,
}
EXPECTED_COUNTS = {}
EXPECTED_SC_CATEGORIES = None


def target_counts_for_total(total):
    """Return dynamic integer 70/15/15 targets with largest-remainder rounding."""
    exact = {split: total * SPLIT_RATIOS[split] for split in SPLITS}
    targets = {split: int(math.floor(exact[split])) for split in SPLITS}
    remainder = total - sum(targets.values())
    order = sorted(SPLITS, key=lambda split: (-(exact[split] - targets[split]), SPLITS.index(split)))
    for split in order[:remainder]:
        targets[split] += 1
    if total >= len(SPLITS):
        for split in SPLITS:
            if targets[split] == 0:
                donor = max(SPLITS, key=lambda name: targets[name])
                targets[donor] -= 1
                targets[split] += 1
    return targets


def assign_groups_to_splits(groups, rng, require_all_splits=True):
    """Assign indivisible groups near dynamic file-count targets."""
    prepared = []
    for group in groups:
        files = sorted(list(group["files"]), key=lambda path: str(path).lower())
        if files:
            prepared.append({**group, "files": files, "size": len(files)})
    total = sum(group["size"] for group in prepared)
    targets = target_counts_for_total(total)
    assignments = {split: [] for split in SPLITS}
    counts = Counter()
    if not prepared:
        return assignments, targets, {split: 0 for split in SPLITS}

    order = list(range(len(prepared)))
    rng.shuffle(order)
    pending = [prepared[index] for index in order]
    if require_all_splits and len(pending) >= len(SPLITS):
        for split in sorted(SPLITS, key=lambda name: (targets[name], SPLITS.index(name))):
            index = min(
                range(len(pending)),
                key=lambda i: (pending[i]["size"], str(pending[i]["files"][0]).lower()),
            )
            group = pending.pop(index)
            assignments[split].append(group)
            counts[split] += group["size"]

    for group in pending:
        def score(split):
            next_counts = {name: counts[name] for name in SPLITS}
            next_counts[split] += group["size"]
            error = sum(abs(next_counts[name] - targets[name]) for name in SPLITS)
            deficit = (targets[split] - counts[split]) / max(targets[split], 1)
            overshoot = max(0, next_counts[split] - targets[split])
            return (error, -deficit, overshoot, counts[split], SPLITS.index(split))
        split = min(SPLITS, key=score)
        assignments[split].append(group)
        counts[split] += group["size"]

    return assignments, targets, {split: counts[split] for split in SPLITS}


def file_sha256(filepath):
    """Compute SHA-256 hash of a file."""
    h = hashlib.sha256()
    with open(filepath, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def get_wav_info(filepath):
    """Get sample_rate, channels, duration from a WAV file."""
    try:
        with wave.open(str(filepath), "rb") as wf:
            return wf.getframerate(), wf.getnchannels(), wf.getnframes() / wf.getframerate()
    except Exception:
        return None, None, None


def split_source_groups(source_map, rng, label):
    """Split whole source recordings near dynamic clip-count targets."""
    groups = [{"source_id": name, "files": files} for name, files in sorted(source_map.items())]
    targets = target_counts_for_total(sum(len(group["files"]) for group in groups))
    if len(groups) < len(SPLITS):
        assignments = {split: [] for split in SPLITS}
        assignments["train"] = groups
        counts = {"train": sum(len(group["files"]) for group in groups), "validation": 0, "test": 0}
        coverage_possible = False
        print(f"Only {len(groups)} {label.lower()} source(s); all clips remain in train.")
    else:
        assignments, targets, counts = assign_groups_to_splits(groups, rng, require_all_splits=True)
        coverage_possible = True

    records = []
    for split in SPLITS:
        for group in assignments[split]:
            for filepath in group["files"]:
                records.append({
                    "filepath": filepath,
                    "source_recording_id": group["source_id"],
                    "split": split,
                })
    return records, targets, counts, coverage_possible


print("Configuration loaded.")
print(f"  Split ratios: train={TRAIN_RATIO}, val={VAL_RATIO}, test={TEST_RATIO}")
print("  Counts are discovered at runtime.")


## 2. Locate Project Root

In [ ]:
# ============================================================
# SECTION 2 - PROJECT ROOT
# ============================================================

def find_project_root():
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "dataset").is_dir() and (candidate / "model_v1").is_dir():
            return candidate
    fallback = Path(r"c:/Users/Mayank Singh/Codes/SIH 2026")
    if fallback.is_dir():
        return fallback
    raise FileNotFoundError("Could not find PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
DATASET_SOURCE = PROJECT_ROOT / "dataset" / "Dataset_v1"
MODEL_V2 = PROJECT_ROOT / "model_v2"
TEMP_DIR = PROJECT_ROOT / "temp"

if not DATASET_SOURCE.is_dir():
    raise FileNotFoundError(f"Missing source dataset: {DATASET_SOURCE}")

POSITIVE_DIR = DATASET_SOURCE / "positive"
SILENCE_DIR = DATASET_SOURCE / "negative_silence"
BACKGROUND_DIR = DATASET_SOURCE / "negative_background"
SPEECH_CMD_DIR = DATASET_SOURCE / "negative_speech_commands"
for directory in [POSITIVE_DIR, SILENCE_DIR, BACKGROUND_DIR, SPEECH_CMD_DIR]:
    if not directory.is_dir():
        raise FileNotFoundError(f"Missing required source directory: {directory}")

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATASET_SOURCE: {DATASET_SOURCE}")
print(f"MODEL_V2: {MODEL_V2}")


In [ ]:
# ── Create V2 directory structure ─────────────────────────────

V2_DIRS = {
    "data":       MODEL_V2 / "data",
    "features":   MODEL_V2 / "features",
    "checkpoints": MODEL_V2 / "checkpoints",
    "evaluation": MODEL_V2 / "evaluation",
    "exports":    MODEL_V2 / "exports",
    "scripts":    MODEL_V2 / "scripts",
    "manifests":  MODEL_V2 / "data" / "manifests",
}

for name, path in V2_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)

for split in SPLITS:
    for cat in CATEGORIES:
        (V2_DIRS["data"] / split / cat).mkdir(parents=True, exist_ok=True)

# Temp directory for any intermediate files
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print("V2 directory structure created.")

## 3. Source Dataset Inspection

In [ ]:
# ============================================================
# SECTION 3 - DISCOVER SOURCE FILES
# ============================================================

def sorted_wavs(root):
    return sorted(
        [path for path in root.rglob("*.wav") if path.is_file()],
        key=lambda path: str(path.relative_to(root)).replace("\\", "/").lower(),
    )


positive_files = sorted_wavs(POSITIVE_DIR)
silence_files = sorted_wavs(SILENCE_DIR)
background_files = sorted_wavs(BACKGROUND_DIR)
speech_cmd_files = sorted_wavs(SPEECH_CMD_DIR)

for label, files in [
    ("positive", positive_files),
    ("negative_silence", silence_files),
    ("negative_background", background_files),
    ("negative_speech_commands", speech_cmd_files),
]:
    if not files:
        raise RuntimeError(f"No WAV files found for {label} under {DATASET_SOURCE}.")

EXPECTED_COUNTS = {
    "positive": len(positive_files),
    "negative_silence": len(silence_files),
    "negative_background": len(background_files),
    "negative_speech_commands": len(speech_cmd_files),
}
EXPECTED_COUNTS["grand_total"] = sum(EXPECTED_COUNTS.values())

positive_rel_paths = [str(path.relative_to(POSITIVE_DIR)).replace("\\", "/") for path in positive_files]
if len(positive_rel_paths) != len(set(positive_rel_paths)):
    raise RuntimeError("Duplicate positive relative paths discovered.")

print("=" * 60)
print("SOURCE FILE COUNTS (DISCOVERED AT RUNTIME)")
print("=" * 60)
for category in CATEGORIES:
    print(f"  {category:28s}: {EXPECTED_COUNTS[category]:>8,}")
print(f"  {'grand_total':28s}: {EXPECTED_COUNTS['grand_total']:>8,}")


def extract_positive_speaker(filepath):
    parts = filepath.relative_to(POSITIVE_DIR).parts
    if len(parts) > 1:
        return parts[0]
    stem_parts = filepath.stem.split("_")
    if len(stem_parts) > 1 and stem_parts[-1].isdigit():
        return "_".join(stem_parts[:-1])
    return "unknown"


def extract_source(filepath, root):
    parts = filepath.relative_to(root).parts
    if len(parts) > 1:
        return parts[0]
    for pattern in [r"^(.+?)_(?:clip|segment|chunk)_\d+$", r"^(.+?)_\d{4,}$"]:
        match = re.match(pattern, filepath.stem, flags=re.IGNORECASE)
        if match:
            return match.group(1)
    return filepath.stem


pos_speakers = defaultdict(list)
silence_sources = defaultdict(list)
bg_sources = defaultdict(list)
for filepath in positive_files:
    pos_speakers[extract_positive_speaker(filepath)].append(filepath)
for filepath in silence_files:
    silence_sources[extract_source(filepath, SILENCE_DIR)].append(filepath)
for filepath in background_files:
    bg_sources[extract_source(filepath, BACKGROUND_DIR)].append(filepath)

print("\nPositive speakers:")
for speaker in sorted(pos_speakers):
    print(f"  {speaker:<20s}: {len(pos_speakers[speaker]):>7,}")
print("\nSilence sources:")
for source_name in sorted(silence_sources):
    print(f"  {source_name!r}: {len(silence_sources[source_name]):>7,}")
print("\nBackground sources:")
for source_name in sorted(bg_sources):
    print(f"  {source_name!r}: {len(bg_sources[source_name]):>7,}")


## 4. Audio Validation

Validate that all source files meet requirements:
- Decodable WAV
- 16 kHz sample rate
- Mono (1 channel)
- PCM16 (2-byte samples)
- ~1 second duration (±50ms tolerance)

Files that fail are **reported but not silently deleted**.

In [ ]:
# ============================================================
# SECTION 4 - AUDIO VALIDATION
# ============================================================

def validate_wav(filepath):
    """Validate a single WAV file. Returns dict with properties and issues."""
    result = {
        "filepath": str(filepath),
        "filename": filepath.name,
        "readable": False,
        "sample_rate": None,
        "channels": None,
        "sample_width": None,
        "duration_s": None,
        "issues": [],
    }

    try:
        with wave.open(str(filepath), "rb") as wf:
            result["sample_rate"] = wf.getframerate()
            result["channels"] = wf.getnchannels()
            result["sample_width"] = wf.getsampwidth()
            result["duration_s"] = wf.getnframes() / wf.getframerate()
            result["readable"] = True
    except Exception as e:
        result["issues"].append(f"Cannot read WAV header: {e}")
        return result

    if result["sample_rate"] != EXPECTED_SR:
        result["issues"].append(f"SR={result['sample_rate']} (expected {EXPECTED_SR})")
    if result["channels"] != EXPECTED_CHANNELS:
        result["issues"].append(f"CH={result['channels']} (expected {EXPECTED_CHANNELS})")
    if result["sample_width"] != EXPECTED_SAMPLE_WIDTH:
        result["issues"].append(f"SW={result['sample_width']} (expected {EXPECTED_SAMPLE_WIDTH})")
    if result["duration_s"] is not None:
        if abs(result["duration_s"] - EXPECTED_DURATION_S) > DURATION_TOLERANCE_S:
            result["issues"].append(
                f"Duration={result['duration_s']:.3f}s (expected {EXPECTED_DURATION_S}s)"
            )
    return result


def validate_file_set(files, category_name):
    """Validate every WAV file in a selected file set."""
    results = []
    for f in tqdm(files, desc=f"Validating {category_name}", leave=True):
        r = validate_wav(f)
        r["category"] = category_name
        results.append(r)

    issues_count = sum(1 for r in results if r["issues"])
    unreadable = sum(1 for r in results if not r["readable"])
    print(f"  Total: {len(files)}, Issues: {issues_count}, Unreadable: {unreadable}")

    if issues_count > 0:
        print("  Files with issues:")
        for r in results:
            if r["issues"]:
                print(f"    {r['filename']}: {'; '.join(r['issues'])}")

    return results


print("=" * 60)
print("VALIDATING ALL NON-SPEECH-COMMAND SOURCE FILES")
print("=" * 60)
print()

val_positive = validate_file_set(positive_files, "positive")
print()
val_silence = validate_file_set(silence_files, "negative_silence")
print()
val_background = validate_file_set(background_files, "negative_background")
print()
print("Selected Speech Commands will be validated after stratified selection.")

all_val_results = val_positive + val_silence + val_background
total_issues = sum(1 for r in all_val_results if r["issues"])
total_unreadable = sum(1 for r in all_val_results if not r["readable"])

print()
print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"  Total validated so far: {len(all_val_results)}")
print(f"  Files with issues:     {total_issues}")
print(f"  Unreadable files:      {total_unreadable}")

if total_issues:
    raise RuntimeError("Audio validation failed for source files. Do not continue.")

print("  All validated files pass checks so far.")

## 5. Speech Commands — Category Discovery

In [ ]:
# ============================================================
# SECTION 5 — SPEECH COMMANDS CATEGORY DISCOVERY
# ============================================================

def extract_sc_category(filepath):
    """Extract word/category from Speech Commands filename.
    Format: <word>_<speakerhash>_nohash_<idx>.wav
    Category is everything before the first 8-char hex hash.
    """
    parts = filepath.stem.split('_')
    category_parts = []
    for p in parts:
        if len(p) >= 8 and all(c in '0123456789abcdef' for c in p):
            break
        category_parts.append(p)
    return '_'.join(category_parts) if category_parts else parts[0]

def extract_sc_speaker(filepath):
    """Extract speaker hash from Speech Commands filename."""
    parts = filepath.stem.split('_')
    for p in parts:
        if len(p) == 8 and all(c in '0123456789abcdef' for c in p):
            return p
    return "unknown"

# Build category → files mapping
sc_categories = defaultdict(list)
for f in speech_cmd_files:
    cat = extract_sc_category(f)
    sc_categories[cat].append(f)
sc_categories = dict(sorted(sc_categories.items()))

print("=" * 60)
print("SPEECH COMMANDS — CATEGORIES")
print("=" * 60)
print(f"Total categories discovered: {len(sc_categories)}")
print(f"Total files: {len(speech_cmd_files):,}")
print()
print(f"{'Category':<20s} {'Count':>8s}")
print("─" * 30)
for cat, files in sc_categories.items():
    print(f"{cat:<20s} {len(files):>8,}")
print("─" * 30)
print(f"{'TOTAL':<20s} {sum(len(v) for v in sc_categories.values()):>8,}")

## 6. Speech Commands - Dynamic Category Split

All discovered Speech Commands WAVs are included; there is no fixed per-category cap.


In [ ]:
# %% [markdown]
# ## 6. Speech Commands - Dynamic, Category-Balanced Selection
#
# Speech Commands negatives are sized relative to the positive class
# (SC_NEGATIVE_RATIO × number of positives) rather than using every
# discovered file, and the target is spread evenly across all discovered
# word categories so no single category dominates the negative class.

# %%
# ============================================================
# SECTION 6 - SPEECH COMMANDS: DYNAMIC, BALANCED SELECTION
# ============================================================

SC_NEGATIVE_RATIO = 10

sc_category_sizes = {category: len(files) for category, files in sc_categories.items()}
sc_total_available = sum(sc_category_sizes.values())
sc_target_total = min(len(positive_files) * SC_NEGATIVE_RATIO, sc_total_available)

print("=" * 60)
print("SPEECH COMMANDS - DYNAMIC TARGET")
print("=" * 60)
print(f"Positives discovered:        {len(positive_files):,}")
print(f"SC_NEGATIVE_RATIO:           {SC_NEGATIVE_RATIO}")
print(f"Computed target (uncapped):  {len(positive_files) * SC_NEGATIVE_RATIO:,}")
print(f"Speech Commands available:   {sc_total_available:,}")
print(f"Speech Commands target used: {sc_target_total:,}")
if sc_target_total < len(positive_files) * SC_NEGATIVE_RATIO:
    print("  NOTE: capped by availability - using every discovered file.")


def allocate_balanced_sc_targets(total_target, category_sizes):
    """
    Distribute `total_target` files across categories as evenly as
    possible without exceeding any category's available count.
    Deterministic: remainder units go to categories in alphabetical
    order. Shortfall from undersized categories is redistributed to
    categories that still have room, in successive passes.
    """
    categories = sorted(category_sizes.keys())
    available = dict(category_sizes)
    allocated = {cat: 0 for cat in categories}
    remaining = min(total_target, sum(available.values()))
    open_categories = [cat for cat in categories if available[cat] > 0]

    while remaining > 0 and open_categories:
        n_open = len(open_categories)
        base = remaining // n_open
        extra = remaining % n_open
        round_target = {
            cat: base + (1 if idx < extra else 0)
            for idx, cat in enumerate(open_categories)
        }

        exhausted = []
        for cat in open_categories:
            room = available[cat] - allocated[cat]
            grant = min(round_target[cat], room)
            allocated[cat] += grant
            remaining -= grant
            if allocated[cat] >= available[cat]:
                exhausted.append(cat)

        if not exhausted:
            break
        open_categories = [cat for cat in open_categories if cat not in exhausted]

    return allocated, remaining


sc_targets, sc_unmet = allocate_balanced_sc_targets(sc_target_total, sc_category_sizes)

print(f"\n{'Category':<20s} {'Available':>10s} {'Target':>8s}")
print("-" * 42)
for category in sorted(sc_targets):
    print(f"{category:<20s} {sc_category_sizes[category]:>10,} {sc_targets[category]:>8,}")
print("-" * 42)
print(f"{'TOTAL':<20s} {sc_total_available:>10,} {sum(sc_targets.values()):>8,}")
if sc_unmet > 0:
    print(f"\nWARNING: {sc_unmet} of the target could not be allocated (all categories exhausted).")


def select_balanced_speech_commands(category_files, targets, seed):
    """
    For each category, take a seeded shuffle of its files and walk it in
    order, skipping any file whose SHA-256 duplicates one already chosen
    in that category, until the category's target count is reached (or
    its unique-audio files run out).
    """
    rng = np.random.RandomState(seed)
    selected = {}
    hash_by_file = {}

    for category in sorted(category_files.keys()):
        files = sorted(category_files[category], key=lambda p: p.name)
        shuffled = list(files)
        rng.shuffle(shuffled)

        target = targets[category]
        seen_hashes = set()
        chosen = []

        for f in shuffled:
            if len(chosen) >= target:
                break
            h = file_sha256(f)
            hash_by_file[f] = h
            if h in seen_hashes:
                continue
            seen_hashes.add(h)
            chosen.append(f)

        if len(chosen) < target:
            print(
                f"  NOTE: {category} yielded only {len(chosen)} unique-audio "
                f"files (target was {target}); using all of them."
            )

        selected[category] = sorted(chosen, key=lambda p: p.name)

    return selected, hash_by_file


print("\nSelecting files per category (SHA-256 duplicate-aware, seeded)...")
selected_sc, selected_sc_hash_by_file = select_balanced_speech_commands(
    sc_categories, sc_targets, SEED
)

EXPECTED_SC_CATEGORIES = len(sc_categories)
selected_sc_files = [
    (filepath, category)
    for category, files in selected_sc.items()
    for filepath in files
]

# Update expected counts to reflect the dynamic selection, not the full
# discovered pool - downstream checks compare against these values.
EXPECTED_COUNTS["negative_speech_commands_discovered"] = EXPECTED_COUNTS["negative_speech_commands"]
EXPECTED_COUNTS["negative_speech_commands"] = len(selected_sc_files)
EXPECTED_COUNTS["grand_total"] = (
    EXPECTED_COUNTS["positive"]
    + EXPECTED_COUNTS["negative_silence"]
    + EXPECTED_COUNTS["negative_background"]
    + EXPECTED_COUNTS["negative_speech_commands"]
)

print(f"\nSpeech Commands categories discovered: {EXPECTED_SC_CATEGORIES}")
print(f"Speech Commands files selected:        {len(selected_sc_files):,} "
      f"(of {EXPECTED_COUNTS['negative_speech_commands_discovered']:,} discovered)")

selected_sc_paths = [filepath for filepath, _ in selected_sc_files]
val_sc_selected = validate_file_set(selected_sc_paths, "negative_speech_commands")
all_val_results = val_positive + val_silence + val_background + val_sc_selected
if len(all_val_results) != EXPECTED_COUNTS["grand_total"]:
    raise RuntimeError("Not every selected source file was validated.")
if any(result["issues"] for result in all_val_results):
    raise RuntimeError("Audio validation failed for selected files.")

selection_df = pd.DataFrame([
    {"filepath": str(filepath), "category": category}
    for filepath, category in selected_sc_files
])
selection_path = V2_DIRS["manifests"] / "speech_commands_selection.csv"
selection_df.to_csv(selection_path, index=False)
print(f"Saved Speech Commands selection: {selection_path}")

## 7. Positive Split - Per-Speaker, SHA-Aware 70/15/15

Every discovered positive WAV file must appear in the manifest exactly once.
Byte-identical positive WAVs are kept, but each identical-audio SHA-256 group
is assigned to one split only so identical audio cannot cross train,
validation, and test.

In [ ]:
# ============================================================
# SECTION 7 - POSITIVE SPLIT (PER-SPEAKER, SHA-AWARE)
# ============================================================

positive_records = []
positive_hash_by_file = {}

print("=" * 60)
print("POSITIVE SPLIT - PER-SPEAKER, SHA-AWARE")
print("=" * 60)
print()
print("Hashing all discovered positive WAV files before splitting...")
for f in tqdm(positive_files, desc="Hashing positive WAVs"):
    positive_hash_by_file[f] = file_sha256(f)

positive_hash_groups = defaultdict(list)
for f, h in positive_hash_by_file.items():
    positive_hash_groups[h].append(f)

duplicate_positive_hashes = {
    h: files for h, files in positive_hash_groups.items() if len(files) > 1
}
cross_speaker_hashes = []
for h, files in duplicate_positive_hashes.items():
    speakers = sorted({extract_positive_speaker(f) for f in files})
    if len(speakers) > 1:
        cross_speaker_hashes.append((h, speakers, files))

print(f"Positive WAV files discovered:       {len(positive_files)}")
print(f"Unique positive SHA-256 groups:      {len(positive_hash_groups)}")
print(f"Duplicate positive SHA-256 groups:   {len(duplicate_positive_hashes)}")
print("Duplicate positive audio is kept and assigned as indivisible split groups.")

if cross_speaker_hashes:
    print("Cross-speaker positive duplicate hashes found:")
    for h, speakers, files in cross_speaker_hashes[:10]:
        names = [f.name for f in files]
        print(f"  {h[:16]}... speakers={speakers} files={names}")
    raise RuntimeError(
        "Positive duplicate hashes span speakers. Resolve speaker/source identity "
        "before per-speaker splitting."
    )

print()
print(f"{'Speaker':<12s} {'Files':>6s} {'Groups':>6s} {'Train':>6s} {'Val':>6s} {'Test':>6s}  "
      f"{'Target train/val/test':>24s}")
print("-" * 86)

# Fail early: three distinct recordings are required for speaker coverage.
for speaker, files in sorted(pos_speakers.items()):
    n_unique = len({positive_hash_by_file[filepath] for filepath in files})
    if n_unique < 3:
        raise RuntimeError(
            f"Speaker {speaker!r} has only {n_unique} unique recording(s) "
            f"({len(files)} files) - cannot guarantee train/val/test coverage. "
            "Add more recordings for this speaker or drop them from the dataset."
        )

rng_pos = np.random.RandomState(SEED)

for speaker in sorted(pos_speakers.keys()):
    files = pos_speakers[speaker]
    hash_groups = defaultdict(list)
    for f in files:
        hash_groups[positive_hash_by_file[f]].append(f)

    groups = [
        {"sha256": h, "files": grouped_files}
        for h, grouped_files in sorted(
            hash_groups.items(),
            key=lambda item: str(sorted(item[1], key=lambda p: p.name)[0]).lower(),
        )
    ]

    assignments, targets, counts = assign_groups_to_splits(groups, rng_pos, require_all_splits=True)

    for split in SPLITS:
        for group in assignments[split]:
            for f in group["files"]:
                positive_records.append({
                    "filepath": f,
                    "speaker": speaker,
                    "split": split,
                    "sha256": group["sha256"],
                })

    print(
        f"{speaker:<12s} {len(files):>6d} {len(groups):>6d} "
        f"{counts.get('train', 0):>6d} {counts.get('validation', 0):>6d} "
        f"{counts.get('test', 0):>6d}  "
        f"{targets['train']:>6d}/{targets['validation']:>3d}/{targets['test']:<3d}"
    )

totals = Counter(r["split"] for r in positive_records)
print("-" * 86)
print(
    f"{'TOTAL':<12s} {len(positive_records):>6d} {'':>6s} "
    f"{totals['train']:>6d} {totals['validation']:>6d} {totals['test']:>6d}"
)
print()
print(f"positive train + validation + test == {len(positive_records)}")

if len(positive_records) != len(positive_files):
    raise RuntimeError(
        f"Positive split lost files: {len(positive_records)} != {EXPECTED_COUNTS['positive']}"
    )

speaker_split_ok = True
for speaker in sorted(pos_speakers):
    speaker_splits = {r["split"] for r in positive_records if r["speaker"] == speaker}
    if speaker_splits != set(SPLITS):
        speaker_split_ok = False
        print(f"Speaker {speaker!r} appears only in {speaker_splits}")

if not speaker_split_ok:
    raise RuntimeError("Positive speaker representation failed.")

positive_hash_splits = defaultdict(set)
for rec in positive_records:
    positive_hash_splits[rec["sha256"]].add(rec["split"])

positive_cross_split_hashes = {
    h: splits for h, splits in positive_hash_splits.items() if len(splits) > 1
}
if positive_cross_split_hashes:
    raise RuntimeError(
        f"Positive identical audio crosses splits for {len(positive_cross_split_hashes)} hashes."
    )

print("Every positive speaker appears in train, validation, and test.")
print("No positive SHA-256 group crosses train/validation/test.")

print()
print("=" * 60)
print("EXACT POSITIVE FILENAMES BY SPLIT")
print("=" * 60)
for split in SPLITS:
    split_names = sorted(str(r["filepath"].relative_to(POSITIVE_DIR)).replace("\\", "/") for r in positive_records if r["split"] == split)
    print(f"\n{split.upper()} positive files ({len(split_names)}):")
    for name in split_names:
        print(f"  {name}")

## 8. Silence and Background - Source-Aware Dynamic Split

Whole source recordings are assigned near the dynamic 70/15/15 clip targets. If there are fewer than three independent sources, all clips stay in train.


In [ ]:
# ============================================================
# SECTION 8 - SILENCE SOURCE-AWARE SPLIT
# ============================================================

print("SILENCE - SOURCE-AWARE DYNAMIC SPLIT")
silence_records, silence_targets, silence_counts, silence_coverage_possible = split_source_groups(
    silence_sources, np.random.RandomState(SEED + 1), "Silence"
)
print(f"Dynamic clip targets: {silence_targets}")
print(f"Actual split counts: {silence_counts}")


In [ ]:
# ============================================================
# SECTION 8 - BACKGROUND SOURCE-AWARE SPLIT
# ============================================================

print("BACKGROUND - SOURCE-AWARE DYNAMIC SPLIT")
bg_records, bg_targets, bg_counts, bg_coverage_possible = split_source_groups(
    bg_sources, np.random.RandomState(SEED + 2), "Background"
)
print(f"Dynamic clip targets: {bg_targets}")
print(f"Actual split counts: {bg_counts}")


In [ ]:
# ============================================================
# SPEECH COMMANDS - DYNAMIC, STRATIFIED, SHA-AWARE SPLIT
# ============================================================

sc_records = []
# selected_sc_hash_by_file was already computed during balanced selection
# (Section 6) - only hash anything that's somehow missing.
for filepath, _ in tqdm(selected_sc_files, desc="Verifying Speech Commands hashes"):
    if filepath not in selected_sc_hash_by_file:
        selected_sc_hash_by_file[filepath] = file_sha256(filepath)

rng_sc_split = np.random.RandomState(SEED + 3)
for category in sorted(selected_sc):
    files = selected_sc[category]
    hash_groups = defaultdict(list)
    for filepath in files:
        hash_groups[selected_sc_hash_by_file[filepath]].append(filepath)
    if len(hash_groups) < 3:
        raise RuntimeError(
            f"Speech Commands category {category!r} has only {len(hash_groups)} unique groups; "
            "cannot guarantee train/validation/test representation."
        )
    groups = [{"sha256": sha256, "files": group_files} for sha256, group_files in hash_groups.items()]
    assignments, targets, counts = assign_groups_to_splits(groups, rng_sc_split, require_all_splits=True)
    print(f"{category:<24s}: targets={targets}, actual={counts}")
    for split in SPLITS:
        for group in assignments[split]:
            for filepath in group["files"]:
                sc_records.append({
                    "filepath": filepath,
                    "subcategory": category,
                    "split": split,
                    "sha256": group["sha256"],
                })

if len(sc_records) != len(selected_sc_files):
    raise RuntimeError("Speech Commands split lost files.")

sc_hash_splits = defaultdict(set)
for record in sc_records:
    sc_hash_splits[record["sha256"]].add(record["split"])
if any(len(splits) > 1 for splits in sc_hash_splits.values()):
    raise RuntimeError("Speech Commands identical audio crosses splits.")

sc_df_check = pd.DataFrame(sc_records)
for category, files in selected_sc.items():
    category_df = sc_df_check[sc_df_check["subcategory"] == category]
    if len(category_df) != len(files) or set(category_df["split"]) != set(SPLITS):
        raise RuntimeError(f"Speech Commands category representation failed for {category!r}.")
print(f"Speech Commands category balance preserved for {len(selected_sc)} categories.")


## 9. Dataset Manifest Creation

Build the full manifest with all required fields including SHA-256 hashes.

In [ ]:
# ============================================================
# SECTION 9 - BUILD FULL MANIFEST
# ============================================================

print("Building manifest with SHA-256 hashes...")
print()

manifest_rows = []

print("Hashing positive files...")
for rec in tqdm(positive_records, desc="Positive"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_POSITIVE,
        "category": "positive",
        "subcategory": "vaani",
        "speaker_or_source_id": rec["speaker"],
        "source_recording_id": f.name,
        "split": rec["split"],
        "sha256": rec["sha256"],
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

print("Hashing silence files...")
for rec in tqdm(silence_records, desc="Silence"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_NEGATIVE,
        "category": "negative_silence",
        "subcategory": "silence",
        "speaker_or_source_id": rec["speaker"],
        "source_recording_id": f.name,
        "split": rec["split"],
        "sha256": file_sha256(f),
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

print("Hashing background files...")
for rec in tqdm(bg_records, desc="Background"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_NEGATIVE,
        "category": "negative_background",
        "subcategory": rec["source_recording_id"],
        "speaker_or_source_id": rec["speaker"],
        "source_recording_id": f.name,
        "split": rec["split"],
        "sha256": file_sha256(f),
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

print("Hashing speech command files...")
for rec in tqdm(sc_records, desc="Speech Cmds"):
    f = rec["filepath"]
    sr, ch, dur = get_wav_info(f)
    manifest_rows.append({
        "source_path": str(f),
        "relative_source_path": str(f.relative_to(PROJECT_ROOT)),
        "filename": f.name,
        "label": LABEL_NEGATIVE,
        "category": "negative_speech_commands",
        "subcategory": rec["subcategory"],
        "speaker_or_source_id": extract_sc_speaker(f),
        "source_recording_id": f.name,
        "split": rec["split"],
        "sha256": rec["sha256"],
        "sample_rate": sr,
        "channels": ch,
        "duration": dur,
    })

manifest_df = pd.DataFrame(manifest_rows)
source_roots = {
    "positive": POSITIVE_DIR,
    "negative_silence": SILENCE_DIR,
    "negative_background": BACKGROUND_DIR,
    "negative_speech_commands": SPEECH_CMD_DIR,
}
manifest_df["split_relative_path"] = [
    str(Path(source_path).relative_to(source_roots[category])).replace("\\", "/")
    for source_path, category in zip(manifest_df["source_path"], manifest_df["category"])
]
if manifest_df.duplicated(["category", "split", "split_relative_path"]).any():
    raise RuntimeError("Manifest contains colliding split destination paths.")

print(f"\nManifest built: {len(manifest_df)} rows")
print(f"Columns: {list(manifest_df.columns)}")

if len(manifest_df) != EXPECTED_COUNTS["grand_total"]:
    raise RuntimeError(
        f"Manifest row count mismatch: {len(manifest_df)} != {EXPECTED_COUNTS['grand_total']}"
    )

if manifest_df["source_path"].duplicated().any():
    duplicate_paths = manifest_df[manifest_df["source_path"].duplicated(keep=False)]["source_path"].tolist()
    raise RuntimeError(f"Manifest contains duplicate source_path entries: {duplicate_paths[:10]}")

positive_manifest_df = manifest_df[manifest_df["category"] == "positive"]
positive_discovered = {str(p.resolve()) for p in positive_files}
positive_manifest_paths = [str(Path(p).resolve()) for p in positive_manifest_df["source_path"]]
positive_manifest_set = set(positive_manifest_paths)

if len(positive_manifest_paths) != len(positive_manifest_set):
    raise RuntimeError("Positive manifest entries are not unique by source_path.")

missing_positive = sorted(positive_discovered - positive_manifest_set)
extra_positive = sorted(positive_manifest_set - positive_discovered)
if missing_positive or extra_positive:
    print(f"Missing positive manifest entries: {missing_positive[:10]}")
    print(f"Unexpected positive manifest entries: {extra_positive[:10]}")
    raise RuntimeError("Positive manifest does not match discovered positive files exactly.")

print(
    "Verified every discovered positive file has exactly one unique manifest entry "
    f"({len(positive_manifest_paths)} entries)."
)

In [ ]:
# ============================================================
# SHA-256 DUPLICATE REPORT - NO ROW REMOVAL
# ============================================================
# Duplicate audio is allowed only when every identical SHA-256 group stays
# inside one split. Do not drop rows: all discovered positive and Speech Commands
# selected Speech Commands files must remain in the manifest.

print("=" * 60)
print("SHA-256 DUPLICATE REPORT (NO ROW REMOVAL)")
print("=" * 60)

hash_counts = manifest_df["sha256"].value_counts()
dupe_hashes = list(hash_counts[hash_counts > 1].index)

same_split_duplicate_groups = 0
cross_split_duplicate_groups = []

for h in sorted(dupe_hashes):
    dupe_rows = manifest_df[manifest_df["sha256"] == h]
    splits_in_group = set(dupe_rows["split"])
    if len(splits_in_group) > 1:
        cross_split_duplicate_groups.append((h, dupe_rows))
    else:
        same_split_duplicate_groups += 1

print(f"  Duplicate SHA-256 groups:           {len(dupe_hashes)}")
print(f"  Same-split duplicate groups:        {same_split_duplicate_groups}")
print(f"  Cross-split duplicate groups:       {len(cross_split_duplicate_groups)}")
print(f"  Manifest rows preserved:            {len(manifest_df)}")

if cross_split_duplicate_groups:
    print()
    print("Cross-split duplicate audio examples:")
    for h, rows in cross_split_duplicate_groups[:10]:
        print(f"  Hash {h[:16]}...")
        for _, row in rows[["filename", "category", "split"]].iterrows():
            print(f"    {row['filename']} | {row['category']} | {row['split']}")
    raise RuntimeError("Identical audio crosses train/validation/test. Do not copy or train.")

if dupe_hashes:
    print("  Duplicate audio exists, but all duplicate groups are confined to one split.")
else:
    print("  No duplicate audio hashes found.")

In [ ]:
# %%
# ============================================================
# SAME-AUDIO / CONFLICTING-LABEL CHECK
# ============================================================
# If identical audio (same SHA-256) is filed under more than one label,
# that's a dataset construction error, not a legitimate duplicate -
# fail loudly rather than silently training on contradictory labels.

label_conflicts = (
    manifest_df.groupby("sha256")["label"]
    .nunique()
    .loc[lambda counts: counts > 1]
)

if len(label_conflicts) > 0:
    print("LABEL CONFLICT: identical audio filed under multiple labels:")
    for h in label_conflicts.index:
        conflict_rows = manifest_df[manifest_df["sha256"] == h][
            ["filename", "category", "label", "split"]
        ]
        print(f"\n  Hash {h[:16]}...")
        print(conflict_rows.to_string(index=False))
    raise RuntimeError(
        f"{len(label_conflicts)} SHA-256 hash(es) appear under conflicting labels. "
        "Fix the source data before continuing."
    )

print("No same-audio label conflicts found (every SHA-256 hash maps to exactly one label).")

## 10. Copy Dataset into model_v2/data/

Only copies files — never modifies `dataset/Dataset_v1/`.

In [ ]:
# ============================================================
# SECTION 10 - COPY FILES INTO model_v2/data/
# ============================================================

print("=" * 60)
print("COPYING FILES TO model_v2/data/")
print("=" * 60)

print("Clearing previous copied WAVs from split/category folders...")
removed_existing = 0
for split in SPLITS:
    for cat in CATEGORIES:
        dst_dir = V2_DIRS["data"] / split / cat
        dst_dir.mkdir(parents=True, exist_ok=True)
        for old_wav in dst_dir.rglob("*.wav"):
            old_wav.unlink()
            removed_existing += 1
print(f"  Removed old copied WAVs: {removed_existing}")

copy_count = 0
copy_errors = []

for _, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Copying"):
    src = Path(row["source_path"])
    dst_dir = V2_DIRS["data"] / row["split"] / row["category"]
    relative_destination = Path(row["split_relative_path"])
    if relative_destination.is_absolute() or ".." in relative_destination.parts:
        raise RuntimeError(f"Unsafe split-relative path: {relative_destination}")
    dst = dst_dir / relative_destination
    dst.parent.mkdir(parents=True, exist_ok=True)

    try:
        shutil.copy2(str(src), str(dst))
        copy_count += 1
    except Exception as e:
        copy_errors.append(f"{src.name}: {e}")

print(f"\nCopied: {copy_count} files")
if copy_errors:
    print(f"Copy errors: {len(copy_errors)}")
    for err in copy_errors[:10]:
        print(f"  {err}")
    raise RuntimeError("Copy failed. Do not continue.")

print("No copy errors.")

print()
print("Verification - files in model_v2/data/:")
copy_counts_ok = True
for split in SPLITS:
    for cat in CATEGORIES:
        d = V2_DIRS["data"] / split / cat
        n = len(list(d.rglob("*.wav")))
        expected = len(manifest_df[
            (manifest_df["split"] == split) & (manifest_df["category"] == cat)
        ])
        status = "OK" if n == expected else "FAIL"
        print(f"  {status:4s} {split:12s} / {cat:30s}: {n:>5d} (expected {expected})")
        if n != expected:
            copy_counts_ok = False

if not copy_counts_ok:
    raise RuntimeError("Copied file counts do not match manifest. Do not continue.")

In [ ]:
# ── Save manifest ────────────────────────────────────────────

csv_path = V2_DIRS["manifests"] / "v2_dataset_manifest.csv"
manifest_df.to_csv(csv_path, index=False)
print(f"Saved manifest CSV:  {csv_path}")
print(f"  Rows: {len(manifest_df)}")

json_path = V2_DIRS["manifests"] / "v2_dataset_manifest.json"
manifest_df.to_json(json_path, orient="records", indent=2)
print(f"Saved manifest JSON: {json_path}")

## 11. Leakage Verification

Multi-level leakage checks:
1. **Hash-based**: No identical SHA-256 hash across splits
2. **Filename-based**: No identical filename across splits within same category
3. **Source-recording-based**: No source recording ID appears in multiple splits
4. **Speaker-based (positive)**: Verify speaker coverage is intentional

In [ ]:
# ============================================================
# SECTION 11 - LEAKAGE VERIFICATION
# ============================================================

print("=" * 60)
print("LEAKAGE VERIFICATION")
print("=" * 60)

leakage_found = False

print("\nCheck 1: Global SHA-256 - no identical audio across splits")
for i, s1 in enumerate(SPLITS):
    for s2 in SPLITS[i + 1:]:
        hashes_s1 = set(manifest_df[manifest_df["split"] == s1]["sha256"])
        hashes_s2 = set(manifest_df[manifest_df["split"] == s2]["sha256"])
        overlap = hashes_s1 & hashes_s2
        if overlap:
            print(f"  FAIL {s1} vs {s2}: {len(overlap)} duplicate hashes cross splits")
            leakage_found = True
        else:
            print(f"  OK   {s1} vs {s2}: no hash overlap")

print("\nCheck 2: Destination-relative path uniqueness within each category")
for cat in CATEGORIES:
    cat_df = manifest_df[manifest_df["category"] == cat]
    for i, s1 in enumerate(SPLITS):
        for s2 in SPLITS[i + 1:]:
            fns1 = set(cat_df[cat_df["split"] == s1]["split_relative_path"])
            fns2 = set(cat_df[cat_df["split"] == s2]["split_relative_path"])
            overlap = fns1 & fns2
            if overlap:
                print(f"  FAIL {cat}: {len(overlap)} destination-relative paths in both {s1} and {s2}")
                leakage_found = True
            elif fns1 and fns2:
                print(f"  OK   {cat}: no filename overlap between {s1} and {s2}")

print("\nCheck 3: Silence/background source recordings stay in one split")
for cat in ["negative_silence", "negative_background"]:
    cat_df = manifest_df[manifest_df["category"] == cat]
    source_splits = cat_df.groupby("source_recording_id")["split"].apply(set)
    leaked_sources = {
        source_id: splits
        for source_id, splits in source_splits.items()
        if len(splits) > 1
    }
    if leaked_sources:
        leakage_found = True
        for source_id, splits in leaked_sources.items():
            print(f"  FAIL {cat}: source {source_id!r} appears in {splits}")
    print(f"  {'FAIL' if leaked_sources else 'OK  '} {cat}: {len(source_splits)} sources, {len(leaked_sources)} leaked")

print("\nCheck 4: Positive speaker coverage")
pos_df = manifest_df[manifest_df["category"] == "positive"]
for spk in sorted(pos_df["speaker_or_source_id"].unique()):
    spk_splits = set(pos_df[pos_df["speaker_or_source_id"] == spk]["split"])
    if spk_splits == set(SPLITS):
        print(f"  OK   {spk}: present in all 3 splits")
    else:
        print(f"  FAIL {spk}: only in {spk_splits}")
        leakage_found = True

print("\nCheck 5: Duplicate hashes are same-split only")
hash_counts = manifest_df["sha256"].value_counts()
dupe_hashes = hash_counts[hash_counts > 1]
same_split_dupes = 0
cross_split_dupes = 0
for h in dupe_hashes.index:
    splits = set(manifest_df[manifest_df["sha256"] == h]["split"])
    if len(splits) > 1:
        cross_split_dupes += 1
    else:
        same_split_dupes += 1
print(f"  Duplicate hash groups:     {len(dupe_hashes)}")
print(f"  Same-split duplicate groups: {same_split_dupes}")
print(f"  Cross-split duplicate groups: {cross_split_dupes}")
if cross_split_dupes:
    leakage_found = True

print()
if leakage_found:
    print("LEAKAGE DETECTED - review the issues above.")
    raise RuntimeError("Leakage verification failed. Do not train.")

print("NO LEAKAGE DETECTED - dataset splits are clean.")

## 12. Dataset Statistics

In [ ]:
# ============================================================
# SECTION 12 - DATASET STATISTICS
# ============================================================

print("=" * 70)
print("  V2 DATASET SUMMARY")
print("=" * 70)

print()
for cat in CATEGORIES:
    cat_df = manifest_df[manifest_df["category"] == cat]
    cat_label = {
        "positive": "Positive (Vaani)",
        "negative_silence": "Silence",
        "negative_background": "Background",
        "negative_speech_commands": "Speech Commands",
    }[cat]
    print(f"{cat_label}:")
    for split in SPLITS:
        n = len(cat_df[cat_df["split"] == split])
        print(f"  {split:12s}: {n:>5d}")
    print(f"  {'total':12s}: {len(cat_df):>5d}")
    print()

print("Total:")
for split in SPLITS:
    sdf = manifest_df[manifest_df["split"] == split]
    n_pos = len(sdf[sdf["label"] == LABEL_POSITIVE])
    n_neg = len(sdf[sdf["label"] == LABEL_NEGATIVE])
    print(f"  {split:12s}: {len(sdf):>5d}  (pos: {n_pos:>4d}, neg: {n_neg:>4d})")
print(f"  {'GRAND TOTAL':12s}: {len(manifest_df):>5d}")

print()
print("-" * 70)
print("MANDATORY COUNT VERIFICATION")
print("-" * 70)
category_totals = manifest_df.groupby("category").size().to_dict()
observed_counts = {
    "positive": int(category_totals.get("positive", 0)),
    "negative_silence": int(category_totals.get("negative_silence", 0)),
    "negative_background": int(category_totals.get("negative_background", 0)),
    "negative_speech_commands": int(category_totals.get("negative_speech_commands", 0)),
    "grand_total": int(len(manifest_df)),
}

summary_counts_ok = True
count_lines = [
    ("positive train + validation + test", "positive"),
    ("silence", "negative_silence"),
    ("background", "negative_background"),
    ("speech_commands", "negative_speech_commands"),
    ("grand total", "grand_total"),
]
for label, key in count_lines:
    actual = observed_counts[key]
    expected = EXPECTED_COUNTS[key]
    ok = actual == expected
    summary_counts_ok = summary_counts_ok and ok
    status = "OK" if ok else "FAIL"
    print(f"  {status:4s} {label} == {actual} (expected {expected})")

print()
print("-" * 70)
print("SPEAKERS")
print("-" * 70)
pos_df = manifest_df[manifest_df["category"] == "positive"]
print(f"Number of speakers: {pos_df['speaker_or_source_id'].nunique()}")
print()
for spk in sorted(pos_df["speaker_or_source_id"].unique()):
    spk_df = pos_df[pos_df["speaker_or_source_id"] == spk]
    spk_splits = {split: len(spk_df[spk_df["split"] == split]) for split in SPLITS}
    print(
        f"  {spk:<12s}: total={len(spk_df):>4d}  "
        f"train={spk_splits['train']:>3d}  "
        f"val={spk_splits['validation']:>3d}  "
        f"test={spk_splits['test']:>3d}"
    )

print()
print("-" * 70)
print("SPEECH COMMANDS CATEGORIES")
print("-" * 70)
sc_df = manifest_df[manifest_df["category"] == "negative_speech_commands"]
sc_cats_summary = sc_df.groupby("subcategory").size().sort_index()
print(f"Total categories: {len(sc_cats_summary)}")
for cat_name, cnt in sc_cats_summary.items():
    print(f"  {cat_name:<20s}: {cnt:>4d}")

print()
print("-" * 70)
print("SOURCE RECORDINGS PER SPLIT")
print("-" * 70)
for cat in ["negative_silence", "negative_background"]:
    cat_df = manifest_df[manifest_df["category"] == cat]
    cat_label = "Silence" if "silence" in cat else "Background"
    for split in SPLITS:
        sources = cat_df[cat_df["split"] == split]["source_recording_id"].unique()
        print(f"  {cat_label} {split:12s}: {len(sources)} source(s): {list(sources)}")

print()
print("-" * 70)
print("INVALID FILES")
print("-" * 70)
total_issues_found = sum(1 for r in all_val_results if r["issues"])
if total_issues_found == 0:
    print("  No invalid files found during validation.")
else:
    print(f"  {total_issues_found} files had validation issues.")

print()
print("-" * 70)
print("CLASS BALANCE")
print("-" * 70)
n_pos_total = len(manifest_df[manifest_df["label"] == LABEL_POSITIVE])
n_neg_total = len(manifest_df[manifest_df["label"] == LABEL_NEGATIVE])
ratio = n_neg_total / n_pos_total if n_pos_total > 0 else float("inf")
print(f"  Total positive: {n_pos_total}")
print(f"  Total negative: {n_neg_total}")
print(f"  Ratio positive:negative = 1:{ratio:.2f}")
print("  Intentionally negative-heavy; class imbalance is handled during training.")

print()
print(f"  Random seed: {SEED}")

## 13. Final Sanity Checks

In [ ]:
# ============================================================
# SECTION 13 - FINAL SANITY CHECKS
# ============================================================

print("=" * 60)
print("SANITY CHECKS")
print("=" * 60)

checks_passed = 0
checks_total = 0


def record_check(name, ok, detail=""):
    global checks_passed, checks_total
    checks_total += 1
    if ok:
        checks_passed += 1
        print(f"OK   {name}{': ' + detail if detail else ''}")
    else:
        print(f"FAIL {name}{': ' + detail if detail else ''}")


category_counts = manifest_df.groupby("category").size().to_dict()
mandatory_counts_ok = (
    category_counts.get("positive", 0) == EXPECTED_COUNTS["positive"]
    and category_counts.get("negative_silence", 0) == EXPECTED_COUNTS["negative_silence"]
    and category_counts.get("negative_background", 0) == EXPECTED_COUNTS["negative_background"]
    and category_counts.get("negative_speech_commands", 0) == EXPECTED_COUNTS["negative_speech_commands"]
    and len(manifest_df) == EXPECTED_COUNTS["grand_total"]
)
record_check("Mandatory category and grand-total counts", mandatory_counts_ok)

positive_discovered = {str(p.resolve()) for p in positive_files}
pos_df_check = manifest_df[manifest_df["category"] == "positive"]
positive_manifest_paths = [str(Path(p).resolve()) for p in pos_df_check["source_path"]]
positive_manifest_set = set(positive_manifest_paths)
unique_positive_manifest_ok = (
    len(positive_manifest_paths) == len(positive_manifest_set)
    and positive_manifest_set == positive_discovered
    and len(positive_manifest_paths) == EXPECTED_COUNTS["positive"]
)
record_check("Every discovered positive file has one unique manifest entry", unique_positive_manifest_ok)

all_source_paths = manifest_df["source_path"].tolist()
record_check("Manifest source_path values are globally unique", len(all_source_paths) == len(set(all_source_paths)))

filename_leakage_ok = True
for cat in CATEGORIES:
    cat_df = manifest_df[manifest_df["category"] == cat]
    for i, s1 in enumerate(SPLITS):
        for s2 in SPLITS[i + 1:]:
            fns1 = set(cat_df[cat_df["split"] == s1]["split_relative_path"])
            fns2 = set(cat_df[cat_df["split"] == s2]["split_relative_path"])
            if fns1 & fns2:
                filename_leakage_ok = False
                print(f"  Destination-relative path overlap in {cat}: {s1} vs {s2}: {sorted(fns1 & fns2)[:10]}")
record_check("No destination-relative path leakage across splits", filename_leakage_ok)

unique_labels = set(manifest_df["label"].unique())
record_check("Labels are binary {0, 1}", unique_labels.issubset({0, 1}))

all_speakers_in_all = True
for spk in sorted(pos_df_check["speaker_or_source_id"].unique()):
    spk_splits = set(pos_df_check[pos_df_check["speaker_or_source_id"] == spk]["split"])
    if spk_splits != set(SPLITS):
        all_speakers_in_all = False
        print(f"  Speaker {spk} only in {spk_splits}")
record_check("Every positive speaker appears in all 3 splits", all_speakers_in_all)

source_ok = True
for cat in ["negative_silence", "negative_background"]:
    cat_df = manifest_df[manifest_df["category"] == cat]
    for src_id in cat_df["source_recording_id"].unique():
        src_splits = set(cat_df[cat_df["source_recording_id"] == src_id]["split"])
        if len(src_splits) > 1:
            source_ok = False
            print(f"  {cat} source {src_id!r} appears in multiple splits: {src_splits}")
record_check("Silence/background source-aware splitting preserved", source_ok)

silence_coverage_ok = (
    not silence_coverage_possible
    or set(manifest_df[manifest_df["category"] == "negative_silence"]["split"]) == set(SPLITS)
)
record_check("Silence uses validation/test when independent sources permit it", silence_coverage_ok)

sc_df_final = manifest_df[manifest_df["category"] == "negative_speech_commands"]
sc_repr_ok = True
for category, files in selected_sc.items():
    category_df = sc_df_final[sc_df_final["subcategory"] == category]
    if len(category_df) != len(files) or set(category_df["split"]) != set(SPLITS):
        sc_repr_ok = False
record_check("All discovered Speech Commands categories are represented", sc_repr_ok)

files_ok = True
for split in SPLITS:
    for cat in CATEGORIES:
        d = V2_DIRS["data"] / split / cat
        n_actual = len(list(d.rglob("*.wav")))
        n_expected = len(manifest_df[
            (manifest_df["split"] == split) & (manifest_df["category"] == cat)
        ])
        if n_actual != n_expected:
            files_ok = False
            print(f"  {split}/{cat}: {n_actual} copied files (expected {n_expected})")
record_check("Copied file counts match manifest", files_ok)

hash_dupe_across = False
for i, s1 in enumerate(SPLITS):
    for s2 in SPLITS[i + 1:]:
        h1 = set(manifest_df[manifest_df["split"] == s1]["sha256"])
        h2 = set(manifest_df[manifest_df["split"] == s2]["sha256"])
        overlap = h1 & h2
        if overlap:
            hash_dupe_across = True
            print(f"  {len(overlap)} duplicate SHA-256 hashes between {s1} and {s2}")
record_check("No SHA-256 duplicate audio crosses train/validation/test", not hash_dupe_across)

manifest_paths = {str(Path(p).resolve()) for p in manifest_df["source_path"]}
validated_paths = {str(Path(r["filepath"]).resolve()) for r in all_val_results}
validation_issue_count = sum(1 for r in all_val_results if r["issues"])
validation_ok = (
    validation_issue_count == 0
    and manifest_paths.issubset(validated_paths)
    and len(validated_paths) == EXPECTED_COUNTS["grand_total"]
)
record_check("All selected files were validated", validation_ok)

summary_ok = bool(summary_counts_ok)
record_check("Dataset summary mandatory counts passed", summary_ok)

print(f"\n{'=' * 60}")
print(f"SANITY CHECKS: {checks_passed}/{checks_total} PASSED")
print(f"{'=' * 60}")

dataset_ready = checks_passed == checks_total
if not dataset_ready:
    raise RuntimeError(
        "Mandatory dataset checks failed. Do not run augmentation, feature extraction, or training."
    )

print("All mandatory checks passed.")

In [ ]:
# ============================================================
# FINAL MESSAGE
# ============================================================

if "dataset_ready" not in globals() or not dataset_ready:
    raise RuntimeError("Dataset is not ready. Do not continue to Notebook 02.")

print()
print("=" * 70)
print("  V2 DATASET PREPARATION COMPLETE")
print("=" * 70)
print()
print("Outputs:")
print(f"  Data:      {V2_DIRS['data']}")
print(f"  Manifest:  {csv_path}")
print(f"  Manifest:  {json_path}")
print()
print("Discovered and preserved counts:")
print(f"  positive train + validation + test == {EXPECTED_COUNTS['positive']}")
print(f"  silence == {EXPECTED_COUNTS['negative_silence']}")
print(f"  background == {EXPECTED_COUNTS['negative_background']}")
print(f"  speech_commands == {EXPECTED_COUNTS['negative_speech_commands']}")
print(f"  grand total == {EXPECTED_COUNTS['grand_total']}")
print()
print("Next step is Notebook 02 only after this successful preparation run.")
print(f"Source dataset is read-only: {DATASET_SOURCE}")
print("DO NOT train a model in this notebook.")
print("=" * 70)
print("DATASET READY")